In [21]:
# 사전 준비: import + 데이터 로딩 (02-2.ipynb 앞부분을 한 셀로 압축)
import requests
import pandas as pd
from bs4 import BeautifulSoup
import gdown

# 책 JSON 데이터 다운로드 (한 번만)
gdown.download('https://bit.ly/3q9SZix', '20s_best_book.json', quiet=True)

books_df = pd.read_json('20s_best_book.json')
books = books_df.loc[:, 'no':'isbn13']
top10_books = books.head(10)
top10_books

,no,ranking,bookname,authors,publisher,publication_year,isbn13
0,1,1,우리가 빛의 속도로 갈 수 없다면 :김초엽 소설,지은이: 김초엽,허블,2019,9791190090018
1,2,2,달러구트 꿈 백화점.이미예 장편소설,지은이: 이미예,팩토리나인,2020,9791165341909
2,3,3,지구에서 한아뿐 :정세랑 장편소설,지은이: 정세랑,난다,2019,9791188862290
3,4,4,"시선으로부터, :정세랑 장편소설",지은이: 정세랑,문학동네,2020,9788954672214
4,5,5,아몬드 :손원평 장편소설,지은이: 손원평,창비,2017,9788936434267
5,6,6,피프티 피플 :정세랑 장편소설,지은이: 정세랑,창비,2016,9788936434243
6,7,7,목소리를 드릴게요 :정세랑 소설집,지은이: 정세랑,아작,2020,9791165300005
7,8,8,나미야 잡화점의 기적 :히가시노 게이고 장편소설,지은이: 히가시노 게이고 ;옮긴이: 양윤옥,현대문학,2012,9788972756194
8,9,9,선량한 차별주의자,김지혜 지음,창비,2019,9788936477196
9,10,9,쇼코의 미소 :최은영 소설,지은이: 최은영,문학동네,2016,9788954641630


In [24]:
# 13주차 과제2-1: 무게 정보를 추출하는 함수
def get_book_weight(isbn):
    url = 'http://www.yes24.com/Product/Search?domain=BOOK&query={}'
    r = requests.get(url.format(isbn))
    soup = BeautifulSoup(r.text, 'html.parser')
    prd_info = soup.find('a', attrs={'class':'gd_name'})
    if prd_info == None:
        return ''
    url = 'http://www.yes24.com' + prd_info['href']
    r = requests.get(url)
    soup = BeautifulSoup(r.text, 'html.parser')
    prd_detail = soup.find('div', attrs={'id':'infoset_specific'})
    prd_tr_list = prd_detail.find_all('tr')
    for tr in prd_tr_list:
        if tr.find('th').get_text() == '쪽수, 무게, 크기':
            parts = tr.find('td').get_text().split()
            # parts = ['352쪽', '|', '380g', '|', '152*215*30mm']
            return parts[2] if len(parts) >= 3 else ''
    return ''

In [6]:
get_book_weight(9791190090018)

'496g'

In [11]:
book_weight = top10_books.apply(
    lambda row: get_book_weight(row['isbn13']), axis=1)
book_weight.name = 'weight'
print(book_weight)

0    496g
1    358g
2    296g
3    412g
4    388g
5    512g
6    318g
7        
8    324g
9    406g
Name: weight, dtype: str


In [10]:
top10_with_weight = pd.merge(top10_books, book_weight,
                             left_index=True, right_index=True)
top10_with_weight

,no,ranking,bookname,authors,publisher,publication_year,isbn13,weight
0,1,1,우리가 빛의 속도로 갈 수 없다면 :김초엽 소설,지은이: 김초엽,허블,2019,9791190090018,496g
1,2,2,달러구트 꿈 백화점.이미예 장편소설,지은이: 이미예,팩토리나인,2020,9791165341909,358g
2,3,3,지구에서 한아뿐 :정세랑 장편소설,지은이: 정세랑,난다,2019,9791188862290,296g
3,4,4,"시선으로부터, :정세랑 장편소설",지은이: 정세랑,문학동네,2020,9788954672214,412g
4,5,5,아몬드 :손원평 장편소설,지은이: 손원평,창비,2017,9788936434267,388g
5,6,6,피프티 피플 :정세랑 장편소설,지은이: 정세랑,창비,2016,9788936434243,512g
6,7,7,목소리를 드릴게요 :정세랑 소설집,지은이: 정세랑,아작,2020,9791165300005,318g
7,8,8,나미야 잡화점의 기적 :히가시노 게이고 장편소설,지은이: 히가시노 게이고 ;옮긴이: 양윤옥,현대문학,2012,9788972756194,
8,9,9,선량한 차별주의자,김지혜 지음,창비,2019,9788936477196,324g
9,10,9,쇼코의 미소 :최은영 소설,지은이: 최은영,문학동네,2016,9788954641630,406g


In [23]:
#13주차 숙제2-2

In [14]:
# 과제 (2): 쪽수와 무게를 함께 추출 → Series 반환 시 apply 결과가 DataFrame
def get_page_and_weight(isbn):
    url = 'http://www.yes24.com/Product/Search?domain=BOOK&query={}'
    r = requests.get(url.format(isbn))
    soup = BeautifulSoup(r.text, 'html.parser')
    prd_info = soup.find('a', attrs={'class':'gd_name'})
    if prd_info == None:
        return pd.Series({'page_count': '', 'weight': ''})
    url = 'http://www.yes24.com' + prd_info['href']
    r = requests.get(url)
    soup = BeautifulSoup(r.text, 'html.parser')
    prd_detail = soup.find('div', attrs={'id':'infoset_specific'})
    prd_tr_list = prd_detail.find_all('tr')
    for tr in prd_tr_list:
        if tr.find('th').get_text() == '쪽수, 무게, 크기':
            parts = tr.find('td').get_text().split()
            page = parts[0] if len(parts) >= 1 else ''
            weight = parts[2] if len(parts) >= 3 else ''
            return pd.Series({'page_count': page, 'weight': weight})
    return pd.Series({'page_count': '', 'weight': ''})

In [15]:
get_page_and_weight(9791190090018)

page_count    344쪽
weight        496g
dtype: str

In [16]:
page_weight_df = top10_books.apply(
    lambda row: get_page_and_weight(row['isbn13']), axis=1)
print(page_weight_df)

  page_count weight
0       344쪽   496g
1       300쪽   358g
2       228쪽   296g
3       340쪽   412g
4       264쪽   388g
5       396쪽   512g
6       272쪽   318g
7                  
8       244쪽   324g
9       296쪽   406g


In [18]:
top10_with_page_weight = pd.merge(top10_books, page_weight_df,
                                  left_index=True, right_index=True)
top10_with_page_weight

,no,ranking,bookname,authors,publisher,publication_year,isbn13,page_count,weight
0,1,1,우리가 빛의 속도로 갈 수 없다면 :김초엽 소설,지은이: 김초엽,허블,2019,9791190090018,344쪽,496g
1,2,2,달러구트 꿈 백화점.이미예 장편소설,지은이: 이미예,팩토리나인,2020,9791165341909,300쪽,358g
2,3,3,지구에서 한아뿐 :정세랑 장편소설,지은이: 정세랑,난다,2019,9791188862290,228쪽,296g
3,4,4,"시선으로부터, :정세랑 장편소설",지은이: 정세랑,문학동네,2020,9788954672214,340쪽,412g
4,5,5,아몬드 :손원평 장편소설,지은이: 손원평,창비,2017,9788936434267,264쪽,388g
5,6,6,피프티 피플 :정세랑 장편소설,지은이: 정세랑,창비,2016,9788936434243,396쪽,512g
6,7,7,목소리를 드릴게요 :정세랑 소설집,지은이: 정세랑,아작,2020,9791165300005,272쪽,318g
7,8,8,나미야 잡화점의 기적 :히가시노 게이고 장편소설,지은이: 히가시노 게이고 ;옮긴이: 양윤옥,현대문학,2012,9788972756194,,
8,9,9,선량한 차별주의자,김지혜 지음,창비,2019,9788936477196,244쪽,324g
9,10,9,쇼코의 미소 :최은영 소설,지은이: 최은영,문학동네,2016,9788954641630,296쪽,406g
